# 01b — Why the Spp1 threshold failed, and what to replace it with

**The §5 sweep in notebook 01 killed the raw-count definition.** B cells — which should be
Spp1-silent — were *more* Spp1-positive than macrophages at every cutoff (95.7% vs 80.7% at
>=1 UMI), and 80% of all 2M cells scored positive. Macrophages showed **no enrichment at all**
versus the all-cell rate. A raw `Spp1 >= k` threshold is not selecting Spp1-producing
macrophages; it is selecting something else.

No value of `k` repairs that, so this notebook does not tune the threshold. It asks **why**,
then picks a definition that survives the answer.

### The two hypotheses, and how each is falsified

| # | Hypothesis | Prediction if true | Fix |
|---|---|---|---|
| **H1** | **Depth** — raw UMIs aren't size-corrected, so cells with more total counts get more of everything | B cells have higher `total_counts` than macs; the inversion disappears after CP10K normalization | normalize |
| **H2** | **Spatial spillover** — segmented cells absorb neighbours' transcripts | a cell's Spp1 is predicted by its *neighbourhood's* Spp1; Spp1 tracks region, not lineage | subtract local background |

These are not exclusive and the fixes differ, so §3 and §5 test them separately.

### The gate that may end the project's framing (§4)
If **Cancer_cell** dominates total Spp1, then the "SPP1-CD44 interaction-high clusters" in
your deck are a **tumor**-CD44 axis. Spp1+ macs would be passengers in a niche they don't
drive — a different mechanism from the PDF's. This was notebook 04 §3; §5 makes it urgent
enough to pull forward.

**Deliverable:** four candidate definitions scored on the same controls, and a recommendation.

In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

import scanpy as sc
import spatialdata as spd

from scipy.stats import mannwhitneyu, spearmanr
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)

/home/janzules/mamba_envs/spatial_gpu_py311/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/janzules/mamba_envs/spatial_gpu_py311/lib/python3.11/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


## 1. Config + load

In [2]:
PROJ_DIR  = Path("/coh_labs/yunroseli/Jona/CAR-T")
ZARR_FILE = PROJ_DIR / "data/zarr/fullDataset/processing_Zarr/1_C2l_annotated_CC"
OUT_DIR   = PROJ_DIR / "results/spp1_analysis"
FIG_DIR   = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)

TISSUE_COL, CELL_TYPE_COL = "tissue", "c2l_consolidated"
MAC_LABELS = ["M1_like_Mac", "M2_like_Mac", "Intermediate_Mac"]

MICRONS_PER_PIXEL = 0.34
K_NEIGHBORS   = 15      # neighbourhood for the spillover test
SPILLOVER_TISSUES = 4   # tissues to sample for the spatial test (it is the slow part)
TOP_Q = 0.90            # "high" = top decile among macrophages, for the relative definitions

sdata = spd.read_zarr(ZARR_FILE)
adata = sdata.tables["segmentation_counts"]

counts = adata.layers["counts"]
spp1_i = int(adata.var_names.get_loc("Spp1"))
_c = counts[:, spp1_i]
spp1_raw = np.asarray(_c.todense()).ravel() if sp.issparse(_c) else np.asarray(_c).ravel()

# total counts per cell — the depth confound (H1) lives here
total_counts = np.asarray(counts.sum(axis=1)).ravel() if sp.issparse(counts) else counts.sum(axis=1)

ct = adata.obs[CELL_TYPE_COL].astype(str)
adata.obs["ct"] = ct.values
adata.obs["Spp1_raw"] = spp1_raw
adata.obs["total_counts"] = total_counts
adata.obs["Spp1_cp10k"] = 1e4 * spp1_raw / np.maximum(total_counts, 1)   # depth-corrected
adata.obs["is_mac"] = ct.isin(MAC_LABELS).values

print(f"cells {adata.n_obs:,} | macs {int(adata.obs['is_mac'].sum()):,}")
print(f"total_counts: median={np.median(total_counts):.0f} mean={total_counts.mean():.1f}")

cells 2,088,557 | macs 81,565
total_counts: median=809 mean=1168.6


## 2. H1 — is it depth?

If B cells simply carry more transcripts, they get more Spp1 without expressing any.

In [3]:
depth = (adata.obs.groupby("ct")
         .agg(n=("total_counts", "size"),
              median_total=("total_counts", "median"),
              mean_total=("total_counts", "mean"),
              mean_Spp1_raw=("Spp1_raw", "mean"),
              mean_Spp1_cp10k=("Spp1_cp10k", "mean"),
              pct_Spp1_raw_ge2=("Spp1_raw", lambda s: 100 * (s >= 2).mean()))
         .sort_values("mean_Spp1_cp10k", ascending=False))
print("=== depth + Spp1 by cell type (sorted by DEPTH-CORRECTED Spp1) ===")
print(depth.round(2).to_string())

mac_d = depth.loc[[m for m in MAC_LABELS if m in depth.index], "median_total"].median()
b_d   = depth.loc["B", "median_total"] if "B" in depth.index else np.nan
r, p = spearmanr(total_counts, spp1_raw)
print(f"""
median total_counts: macrophages={mac_d:.0f} | B cells={b_d:.0f}  (ratio {b_d/mac_d:.2f}x)
Spearman(total_counts, Spp1_raw) = {r:.3f}

H1 VERDICT
  B depth >> mac depth AND high correlation -> the inversion is a DEPTH artifact; use Spp1_cp10k.
  B depth ~ mac depth                       -> depth does NOT explain it; H2 (spillover) is live.
  Note the sort order above: if macrophages rise to the top once depth-corrected, that alone
  largely rescues the definition.
""")

=== depth + Spp1 by cell type (sorted by DEPTH-CORRECTED Spp1) ===
                        n  median_total   mean_total  mean_Spp1_raw  mean_Spp1_cp10k  pct_Spp1_raw_ge2
ct                                                                                                    
Treg                 5155        1468.0  1866.520020          20.07        92.589996             92.30
Cancer_cell        363019        1377.0  1759.709961          15.85        88.769997             95.27
N2_like_Neu        193240        1228.0  1697.469971          16.66        85.709999             89.82
N1_like_Neu        134656        1283.0  1575.489990          13.32        72.739998             83.80
Erythrocyte        210019        1202.0  1534.319946          12.20        71.660004             86.75
B                   40422        1346.0  1810.000000          12.35        63.919998             89.50
NKT                  7110         999.0  1209.089966           8.04        62.720001             86.22
CD4_T 

## 3. THE GATE — who actually makes Spp1?

`pct_of_total_Spp1` is each cell type's share of the tissue's entire Spp1 signal.

In [4]:
src = (adata.obs.groupby("ct")
       .agg(n_cells=("Spp1_raw", "size"), total_Spp1=("Spp1_raw", "sum"),
            mean_Spp1_cp10k=("Spp1_cp10k", "mean"))
       )
src["pct_of_total_Spp1"] = 100 * src["total_Spp1"] / src["total_Spp1"].sum()
src["pct_of_cells"] = 100 * src["n_cells"] / src["n_cells"].sum()
src["Spp1_per_cell_index"] = src["pct_of_total_Spp1"] / src["pct_of_cells"]   # >1 = punches above weight
src = src.sort_values("pct_of_total_Spp1", ascending=False)
src.to_csv(OUT_DIR / "spp1_source_by_celltype.csv")

print("=== who makes Spp1? ===")
print(src.round(2).to_string())

mac_share = src.loc[[m for m in MAC_LABELS if m in src.index], "pct_of_total_Spp1"].sum()
top = src.index[0]
print(f"""
Macrophages produce {mac_share:.1f}% of all Spp1 transcript.
Largest single source: {top} ({src.loc[top,'pct_of_total_Spp1']:.1f}% of Spp1, {src.loc[top,'pct_of_cells']:.1f}% of cells)

GATE
  mac share >30%  -> macrophage-driven axis; the deck's SPP1-CD44 framing survives.
  mac share <10% and Cancer_cell on top -> a TUMOR-Cd44 axis. Spp1+ macs sit in a niche they
      do not drive. The project is still viable, but the claim must be reworded and the PDF's
      mechanism is not what your data shows.
  Read 'Spp1_per_cell_index' too: >1 means that type over-produces Spp1 for its abundance.
""")

=== who makes Spp1? ===
                   n_cells   total_Spp1  mean_Spp1_cp10k  pct_of_total_Spp1  pct_of_cells  Spp1_per_cell_index
ct                                                                                                            
Cancer_cell         363019  5755021.000        88.769997              31.24         17.38                 1.80
N2_like_Neu         193240  3219379.750        85.709999              17.48          9.25                 1.89
Erythrocyte         210019  2562004.000        71.660004              13.91         10.06                 1.38
Unknown             671434  1915895.000        42.169998              10.40         32.15                 0.32
N1_like_Neu         134656  1793801.875        72.739998               9.74          6.45                 1.51
M2_like_Mac          60720   518775.000        52.810001               2.82          2.91                 0.97
B                    40422   499065.000        63.919998               2.71          1.9

## 4. H2 — spatial spillover

The decisive test. For sampled tissues, regress each cell's Spp1 on the **mean Spp1 of its
`K_NEIGHBORS` nearest neighbours, excluding itself**. Strong dependence = Spp1 is a property
of *where a cell is*, not *what it is* — which is exactly what would fake our result.

Then, the fix: `Spp1_excess = own - neighbourhood mean`. Cells positive on excess produce
Spp1 **above their local background**, which is what "Spp1+ macrophage" should have meant.

In [5]:
def scale_of(t):
    try:
        from spatialdata.transformations import get_transformation
        k = f"{t}_cell_boundaries"
        if k in sdata.shapes:
            return float(get_transformation(sdata.shapes[k], get_all=True)["downscale_to_hires"].scale[0])
    except Exception:
        pass
    return 1.0

tissues = sorted(adata.obs[TISSUE_COL].astype(str).unique())
sample_t = tissues[:SPILLOVER_TISSUES]
adata.obs["Spp1_nbr"] = np.nan

rows = []
for t in sample_t:
    m = adata.obs[TISSUE_COL].astype(str).eq(t).values
    coords = np.asarray(adata.obsm["spatial"][m], float) * scale_of(t) * MICRONS_PER_PIXEL
    own = adata.obs.loc[m, "Spp1_cp10k"].values
    nn = NearestNeighbors(n_neighbors=K_NEIGHBORS + 1).fit(coords)
    _, idx = nn.kneighbors(coords)
    nbr = own[idx[:, 1:]].mean(axis=1)          # drop self
    adata.obs.loc[adata.obs_names[m], "Spp1_nbr"] = nbr
    r_all, _ = spearmanr(own, nbr)
    mac_t = adata.obs.loc[m, "is_mac"].values
    r_mac, _ = spearmanr(own[mac_t], nbr[mac_t]) if mac_t.sum() > 100 else (np.nan, np.nan)
    rows.append({"tissue": t, "n": int(m.sum()),
                 "spearman_own_vs_nbr_allcells": r_all,
                 "spearman_own_vs_nbr_macs": r_mac})
    print(f"  {t}: rho(own, neighbourhood) = {r_all:.3f} (all) | {r_mac:.3f} (macs)", flush=True)

spill = pd.DataFrame(rows)
spill.to_csv(OUT_DIR / "spp1_spillover_test.csv", index=False)
print("\n=== H2: spatial spillover test ===")
print(spill.round(3).to_string(index=False))
print(f"""
mean rho (all cells) = {spill['spearman_own_vs_nbr_allcells'].mean():.3f}

H2 VERDICT
  rho > 0.5  -> Spp1 is strongly REGIONAL. A cell's Spp1 is largely its neighbourhood's Spp1.
                Any 'Spp1+ macs are in the tumour/hypoxic niche' finding built on raw Spp1
                would be circular by construction. Use the local-background-corrected
                definition (D) below.
  rho < 0.2  -> spillover is mild; depth correction (B) is enough.
""")

  CyPSCA_1_1: rho(own, neighbourhood) = 0.550 (all) | 0.661 (macs)
  CyPSCA_1_2: rho(own, neighbourhood) = 0.735 (all) | 0.733 (macs)
  CyPSCA_1_3: rho(own, neighbourhood) = 0.727 (all) | 0.687 (macs)
  CyPSCA_1_4: rho(own, neighbourhood) = 0.742 (all) | 0.752 (macs)

=== H2: spatial spillover test ===
    tissue      n  spearman_own_vs_nbr_allcells  spearman_own_vs_nbr_macs
CyPSCA_1_1 108895                         0.550                     0.661
CyPSCA_1_2  55540                         0.735                     0.733
CyPSCA_1_3  33085                         0.727                     0.687
CyPSCA_1_4  27025                         0.742                     0.752

mean rho (all cells) = 0.689

H2 VERDICT
  rho > 0.5  -> Spp1 is strongly REGIONAL. A cell's Spp1 is largely its neighbourhood's Spp1.
                Any 'Spp1+ macs are in the tumour/hypoxic niche' finding built on raw Spp1
                would be circular by construction. Use the local-background-corrected
             

## 5. Four candidate definitions, same controls

- **A_raw** — `Spp1 >= 2` UMI (notebook 01's; shown as the failing baseline)
- **B_cp10k** — top decile of depth-corrected Spp1 **among macrophages**
- **C_module** — top decile of the SPP1-TAM program (Trem2/Gpnmb/Apoe/Fabp5...) among macs
- **D_excess** — top decile of `Spp1_cp10k - neighbourhood mean` among macs (spillover-corrected)

Scored on: **B-cell control rate** (should be LOW — it is the whole point), n selected, and
TAM-module separation. Note honestly: the module cannot validate **C_module** (circular), so
that cell is marked n/a rather than reported as a win.

In [6]:
TAM_GENES = ["Trem2", "Gpnmb", "Apoe", "Fabp5", "Marco", "Cd63", "Lgals3",
             "Ctsb", "Ctsd", "Lpl", "Abca1", "Abcg1", "Cd36", "Msr1", "Plin2"]
tam_present = [g for g in TAM_GENES if g in adata.var_names]
sc.tl.score_genes(adata, gene_list=tam_present, score_name="TAM_module",
                  use_raw=False, random_state=0)
print(f"TAM_module: {len(tam_present)}/{len(TAM_GENES)} genes -> {tam_present}")

mac = adata.obs["is_mac"].values
bcell = (adata.obs["ct"] == "B").values
has_nbr = adata.obs["Spp1_nbr"].notna().values

def top_decile_among(values, universe, q=TOP_Q):
    """Top-q of `values` computed WITHIN `universe` (so the cut is relative to macrophages)."""
    out = np.zeros(len(values), bool)
    v = pd.Series(values)[universe]
    if v.notna().sum() < 100:
        return out
    thr = v.quantile(q)
    out[universe] = (pd.Series(values)[universe] >= thr).values
    return out

adata.obs["Spp1_excess"] = adata.obs["Spp1_cp10k"] - adata.obs["Spp1_nbr"]

CANDIDATES = {
    "A_raw_ge2":  mac & (adata.obs["Spp1_raw"].values >= 2),
    "B_cp10k_q90": top_decile_among(adata.obs["Spp1_cp10k"].values, mac),
    "C_module_q90": top_decile_among(adata.obs["TAM_module"].values, mac),
    "D_excess_q90": top_decile_among(adata.obs["Spp1_excess"].values, mac & has_nbr),
}
# matching control definitions applied to B cells, using the SAME cut rule
CONTROLS = {
    "A_raw_ge2":   bcell & (adata.obs["Spp1_raw"].values >= 2),
    "B_cp10k_q90": top_decile_among(adata.obs["Spp1_cp10k"].values, bcell),
    "C_module_q90": top_decile_among(adata.obs["TAM_module"].values, bcell),
    "D_excess_q90": top_decile_among(adata.obs["Spp1_excess"].values, bcell & has_nbr),
}

rows = []
for name, sel in CANDIDATES.items():
    univ = mac & has_nbr if name == "D_excess_q90" else mac
    pos, neg = sel, univ & ~sel
    if name == "C_module_q90":
        delta = np.nan       # circular: the module defines the call
    elif pos.sum() > 50 and neg.sum() > 50:
        a = adata.obs.loc[pos, "TAM_module"].values
        b = adata.obs.loc[neg, "TAM_module"].values
        u = mannwhitneyu(a, b, alternative="two-sided")
        delta = 2 * u.statistic / (len(a) * len(b)) - 1
    else:
        delta = np.nan
    # does it recover Spp1 itself? (fair for C; trivially true for A/B)
    spp1_sep = np.nan
    if name == "C_module_q90" and pos.sum() > 50 and neg.sum() > 50:
        a = adata.obs.loc[pos, "Spp1_cp10k"].values
        b = adata.obs.loc[neg, "Spp1_cp10k"].values
        u = mannwhitneyu(a, b, alternative="two-sided")
        spp1_sep = 2 * u.statistic / (len(a) * len(b)) - 1
    rows.append({
        "definition": name,
        "n_macs_selected": int(sel.sum()),
        "pct_of_macs": 100 * sel.sum() / max(univ.sum(), 1),
        "pct_of_B_control": 100 * CONTROLS[name].sum() / max(bcell.sum(), 1),
        "TAM_module_delta": delta,
        "Spp1_delta_forC": spp1_sep,
    })
cand = pd.DataFrame(rows)
cand.to_csv(OUT_DIR / "spp1_candidate_definitions.csv", index=False)

print("\n=== candidate definitions, same controls ===")
print(cand.round(3).to_string(index=False))
print("""
HOW TO PICK
  pct_of_B_control is the discriminator. A_raw_ge2 will show ~89% - that is the bug.
  A usable definition puts the B-cell control FAR below the macrophage rate.
  Relative (q90) definitions select 10% of macs BY CONSTRUCTION, so n is not evidence -
  only the control rate and the module separation are.
  C_module cannot be scored on the module (circular); judge it by whether it independently
  recovers Spp1 (Spp1_delta_forC) and by its control rate.
""")

TAM_module: 15/15 genes -> ['Trem2', 'Gpnmb', 'Apoe', 'Fabp5', 'Marco', 'Cd63', 'Lgals3', 'Ctsb', 'Ctsd', 'Lpl', 'Abca1', 'Abcg1', 'Cd36', 'Msr1', 'Plin2']

=== candidate definitions, same controls ===
  definition  n_macs_selected  pct_of_macs  pct_of_B_control  TAM_module_delta  Spp1_delta_forC
   A_raw_ge2            52984       64.959            89.496             0.325              NaN
 B_cp10k_q90             8157       10.001            10.002             0.069              NaN
C_module_q90             8157       10.001            10.002               NaN            0.112
D_excess_q90              814       10.001             1.252             0.004              NaN

HOW TO PICK
  pct_of_B_control is the discriminator. A_raw_ge2 will show ~89% - that is the bug.
  A usable definition puts the B-cell control FAR below the macrophage rate.
  Relative (q90) definitions select 10% of macs BY CONSTRUCTION, so n is not evidence -
  only the control rate and the module separation are.


In [7]:
# Overlap: do the candidates agree on WHICH macrophages?
names = list(CANDIDATES)
ov = pd.DataFrame(index=names, columns=names, dtype=float)
for i in names:
    for j in names:
        a, b = CANDIDATES[i], CANDIDATES[j]
        inter, union = (a & b).sum(), (a | b).sum()
        ov.loc[i, j] = inter / union if union else np.nan
print("=== Jaccard overlap between definitions ===")
print(ov.round(3).to_string())
print("\nLow overlap between B_cp10k and D_excess => spillover correction genuinely changes")
print("WHICH cells are called, not just how many. That is the whole argument for D.")

print(f"""
==================== SUMMARY — report these back ====================
1. §2 depth table + the H1 VERDICT lines (is B depth >> mac depth? did macs rise to the top
   of the depth-corrected sort?)
2. §3 'who makes Spp1' table + macrophage share  <- THE GATE
3. §4 spillover rho + H2 VERDICT
4. §5 candidate table + the Jaccard overlap

I will pick the definition from these and rewrite notebook 01 §5-7 plus the SPP1_THRESHOLD
in 02-04 to match. Do not run 02-04 until this is settled - every one of them takes the
Spp1 call as given, and right now that call is 80% of all macrophages.
=====================================================================
""")

=== Jaccard overlap between definitions ===
              A_raw_ge2  B_cp10k_q90  C_module_q90  D_excess_q90
A_raw_ge2         1.000        0.154         0.120         0.015
B_cp10k_q90       0.154        1.000         0.037         0.052
C_module_q90      0.120        0.037         1.000         0.005
D_excess_q90      0.015        0.052         0.005         1.000

Low overlap between B_cp10k and D_excess => spillover correction genuinely changes
WHICH cells are called, not just how many. That is the whole argument for D.

==================== SUMMARY — report these back ====================
1. §2 depth table + the H1 VERDICT lines (is B depth >> mac depth? did macs rise to the top
   of the depth-corrected sort?)
2. §3 'who makes Spp1' table + macrophage share  <- THE GATE
3. §4 spillover rho + H2 VERDICT
4. §5 candidate table + the Jaccard overlap

I will pick the definition from these and rewrite notebook 01 §5-7 plus the SPP1_THRESHOLD
in 02-04 to match. Do not run 02-04 until th